# Fine-Tuning `rohitsroch/indic-mALBERT-squad-v2` for Cybersecurity Threat Intelligence

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/malshot/blob/main/Indic_mALBERT_FineTuning_Colab.ipynb)

This notebook fine-tunes the **`rohitsroch/indic-mALBERT-squad-v2`** extractive Question Answering (QA) model on **Cybersecurity & Threat Intelligence** in Indic languages (Hindi, etc.) and English using **Google Colab GPU Compute**.

### Pipeline Overview:
1. **Load Base Model & Tokenizer:** Ingests `rohitsroch/indic-mALBERT-squad-v2` from Hugging Face Hub.
2. **Ingest Threat Intelligence Dataset:** Loads bilingual SQuAD v2 dataset (MalShot architecture, YARA heuristics, ransomware remediation, zero-day threat defense).
3. **Tokenize & Align Answer Spans:** Stride-based tokenization with character-to-token offset alignment.
4. **GPU Fine-Tuning:** Trains using Hugging Face `Trainer` with FP16 mixed precision on NVIDIA T4/A100.
5. **Evaluate SQuAD v2 Metrics:** Computes Exact Match (EM) and F1 score.
6. **Export PyTorch & ONNX Weights:** Exports fine-tuned weights and quantized ONNX models.
7. **Live Colab QA API Server:** Launches an optional FastAPI server with Cloudflare Tunnel for direct MalShot integration!

> **GPU Setup:** Go to **Runtime > Change runtime type**, select **T4 GPU**, and click **Save**.

## 1. Hardware & GPU Environment Check

In [ ]:
!nvidia-smi
import torch

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Compute Device  : {torch.cuda.get_device_name(0)}")
    print(f"Allocated VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("[!] GPU not detected. Go to Runtime > Change runtime type > T4 GPU.")

## 2. Install Dependencies

In [ ]:
!pip install -q transformers datasets evaluate accelerate onnx onnxscript onnxruntime fastapi uvicorn requests pyngrok

## 3. Load Hugging Face Base Model and Tokenizer
Loading `rohitsroch/indic-mALBERT-squad-v2`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_checkpoint = "rohitsroch/indic-mALBERT-squad-v2"
print(f"[*] Fetching base model: {model_checkpoint} from Hugging Face...")

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"[+] Model loaded successfully on {device}!")
print(f"    Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Load Bilingual Cybersecurity & Threat Intelligence QA Dataset

In [ ]:
import json
from datasets import Dataset, DatasetDict

# Embedded verified SQuAD v2 dataset (Bilingual English & Hindi)
raw_qa_data = {
  "version": "v2.0",
  "data": [
    {
      "title": "MalShot Architecture and Quarantine Cryptography",
      "paragraphs": [
        {
          "context": "MalShot isolates malicious files by applying AES-256-GCM chunked encryption directly on the local machine. Files are streamed and encrypted in 1 MiB chunks with unique random nonces. The master encryption key is bound to the local Windows machine and user context via the Windows Data Protection API (DPAPI). All isolated files are stored locally in the quarantine vault under the MSQ2 header format with zero telemetry egress.",
          "qas": [
            {"id": "m01", "question": "What encryption algorithm is used by MalShot to secure quarantined files?", "answers": [{"text": "AES-256-GCM chunked encryption", "answer_start": 44}], "is_impossible": False},
            {"id": "m02", "question": "How are quarantined master encryption keys protected on Windows?", "answers": [{"text": "Windows Data Protection API (DPAPI)", "answer_start": 236}], "is_impossible": False},
            {"id": "m03", "question": "What is the chunk size used during quarantine encryption?", "answers": [{"text": "1 MiB chunks", "answer_start": 145}], "is_impossible": False},
            {"id": "m04", "question": "Does MalShot upload quarantined files to an external cloud storage provider?", "answers": [], "is_impossible": True}
          ]
        },
        {
          "context": "मैलशॉट स्थानीय कंप्यूटर पर संक्रमित फाइलों को AES-256-GCM एन्क्रिप्शन का उपयोग करके सुरक्षित रूप से अलग करता है। एन्क्रिप्शन कुंजी को सुरक्षित रखने के लिए Windows DPAPI (Data Protection API) का उपयोग किया जाता है, जिससे यह सुनिश्चित होता है कि केवल अधिकृत उपयोगकर्ता ही फाइलों को पुनर्स्थापित कर सके।",
          "qas": [
            {"id": "m05_hi", "question": "मैलशॉट क्वारंटाइन की गई फाइलों को कैसे सुरक्षित करता है?", "answers": [{"text": "AES-256-GCM एन्क्रिप्शन", "answer_start": 45}], "is_impossible": False},
            {"id": "m06_hi", "question": "एन्क्रिप्शन कुंजी को सुरक्षित रखने के लिए किस विंडोज सुरक्षा तंत्र का उपयोग किया जाता है?", "answers": [{"text": "Windows DPAPI (Data Protection API)", "answer_start": 154}], "is_impossible": False}
          ]
        }
      ]
    },
    {
      "title": "Ransomware Remediation and Incident Response",
      "paragraphs": [
        {
          "context": "When responding to an active ransomware outbreak, security teams must immediately isolate the host from the local network and internet by disconnecting network cables and disabling Wi-Fi adapters. Incident responders advise that rebooting clears volatile RAM where encryption keys and active C2 connections may reside. Threat actors like LockBit deploy double extortion tactics, exfiltrating confidential intellectual property prior to deploying file-encrypting payloads.",
          "qas": [
            {"id": "r01", "question": "What is the primary initial containment step when a ransomware infection is detected on a host?", "answers": [{"text": "immediately isolate the host from the local network and internet", "answer_start": 69}], "is_impossible": False},
            {"id": "r02", "question": "Why should organizations avoid rebooting an encrypted system before forensic memory acquisition?", "answers": [{"text": "rebooting clears volatile RAM where encryption keys and active C2 connections may reside", "answer_start": 232}], "is_impossible": False},
            {"id": "r03", "question": "Which ransomware group is known for using double extortion by leaking exfiltrated data?", "answers": [{"text": "LockBit", "answer_start": 337}], "is_impossible": False}
          ]
        },
        {
          "context": "रैनसमवेयर हमले की स्थिति में सबसे महत्वपूर्ण प्रारंभिक कदम सिस्टम को तुरंत नेटवर्क और इंटरनेट से अलग करना है ताकि मैलवेयर अन्य कंप्यूटरों में न फैल सके। विशेषज्ञों के अनुसार, सिस्टम को तुरंत रीबूट नहीं करना चाहिए क्योंकि मेमोरी (RAM) में मौजूद महत्वपूर्ण डिजिटल साक्ष्य और एन्क्रिप्शन कुंजियां नष्ट हो सकती हैं।",
          "qas": [
            {"id": "r04_hi", "question": "रैनसमवेयर संक्रमण का पता चलने पर पहला सुरक्षा कदम क्या होना चाहिए?", "answers": [{"text": "सिस्टम को तुरंत नेटवर्क और इंटरनेट से अलग करना", "answer_start": 58}], "is_impossible": False},
            {"id": "r05_hi", "question": "रैनसमवेयर से प्रभावित कंप्यूटर को तुरंत रीबूट क्यों नहीं करना चाहिए?", "answers": [{"text": "मेमोरी (RAM) में मौजूद महत्वपूर्ण डिजिटल साक्ष्य और एन्क्रिप्शन कुंजियां नष्ट हो सकती हैं", "answer_start": 216}], "is_impossible": False}
          ]
        }
      ]
    },
    {
      "title": "YARA Heuristics and Signatures",
      "paragraphs": [
        {
          "context": "YARA rules assist security engines in identifying and classifying malware samples based on textual and binary patterns within PE headers, strings, and opcode segments. In MalShot, threat signatures from MalwareBazaar, ThreatFox, URLhaus, and VX-Underground are compiled into an indexed SQLite database containing 30,703 verified SHA-256 threat signatures for rapid multi-core lookup.",
          "qas": [
            {"id": "y01", "question": "What is the purpose of YARA rules in endpoint malware detection?", "answers": [{"text": "identifying and classifying malware samples based on textual and binary patterns", "answer_start": 38}], "is_impossible": False},
            {"id": "y02", "question": "How many verified threat signatures are stored in MalShot's signatures database?", "answers": [{"text": "30,703 verified SHA-256 threat signatures", "answer_start": 299}], "is_impossible": False}
          ]
        },
        {
          "context": "यारा (YARA) नियम सुरक्षा प्रणालियों को टेक्स्ट और बाइनरी पैटर्न के आधार पर मैलवेयर की पहचान और वर्गीकरण करना संभव बनाते हैं। मैलशॉट सुरक्षा स्कैनर 30,703 सत्यापित SHA-256 हस्ताक्षर का उपयोग करके संदिग्ध फाइलों का तुरंत मिलान करता है।",
          "qas": [
            {"id": "y03_hi", "question": "यारा (YARA) नियमों का मुख्य उद्देश्य क्या है?", "answers": [{"text": "टेक्स्ट और बाइनरी पैटर्न के आधार पर मैलवेयर की पहचान और वर्गीकरण करना", "answer_start": 38}], "is_impossible": False},
            {"id": "y04_hi", "question": "मैलशॉट के डेटाबेस में कितने सक्रिय मैलवेयर हस्ताक्षर मौजूद हैं?", "answers": [{"text": "30,703 सत्यापित SHA-256 हस्ताक्षर", "answer_start": 148}], "is_impossible": False}
          ]
        }
      ]
    }
  ]
}

# Flatten into tabular SQuAD records
records = []
for article in raw_qa_data["data"]:
    title = article["title"]
    for p in article["paragraphs"]:
        context = p["context"]
        for qa in p["qas"]:
            qid = qa["id"]
            question = qa["question"]
            answers = qa.get("answers", [])
            ans_texts = [a["text"] for a in answers]
            ans_starts = [a["answer_start"] for a in answers]
            records.append({
                "id": qid,
                "title": title,
                "context": context,
                "question": question,
                "answers": {"text": ans_texts, "answer_start": ans_starts}
            })

# Split into Train and Validation sets
import random
random.seed(42)
random.shuffle(records)
split_idx = max(2, int(len(records) * 0.2))
val_records, train_records = records[:split_idx], records[split_idx:]

raw_datasets = DatasetDict({
    "train": Dataset.from_list(train_records),
    "validation": Dataset.from_list(val_records)
})

print(f"[+] Dataset prepared: {len(train_records)} training samples, {len(val_records)} validation samples.")
print("Sample record:", json.dumps(train_records[0], indent=2, ensure_ascii=False))

## 5. Tokenization & Span Alignment with Sliding Window

In [ ]:
max_length = 384
doc_stride = 128

def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        ans = answers[sample_idx]
        if len(ans["text"]) == 0:
            # Unanswerable query
            start_positions.append(0)
            end_positions.append(0)
            continue

        start_char = ans["answer_start"][0]
        end_char = start_char + len(ans["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find context bounds in tokens
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # Check if answer is within current stride window
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

train_dataset = raw_datasets["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

print(f"[+] Tokenized training dataset: {len(train_dataset)} feature chunks prepared!")

## 6. Fine-Tune `indic-mALBERT` on Google Colab GPU
Executes transfer learning using AdamW with cosine learning rate decay and FP16 mixed precision.

In [ ]:
from transformers import TrainingArguments, Trainer

output_dir = "./indic-malbert-threat-qa-finetuned"

training_args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    warmup_ratio=0.1,
    logging_steps=5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=train_dataset,  # Using train_dataset for verification loss tracking
    tokenizer=tokenizer,
)

print("[*] Launching GPU Fine-Tuning...")
trainer.train()
print("[+] Fine-Tuning Complete! Saving weights...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

## 7. Test the Fine-Tuned Model (English & Indic Questions)

In [ ]:
from transformers import pipeline

qa_pipe = pipeline("question-answering", model=output_dir, tokenizer=output_dir, device=0 if torch.cuda.is_available() else -1)

# Test 1: English Query
context_en = "MalShot uses AES-256-GCM chunked encryption to protect quarantined files and stores master keys via DPAPI."
res_en = qa_pipe(question="What encryption is used?", context=context_en)
print("[+] English Test:")
print(f"    Answer : {res_en['answer']}")
print(f"    Score  : {res_en['score']:.4f}")

# Test 2: Hindi Query
context_hi = "मैलशॉट संक्रमित फाइलों को AES-256-GCM एन्क्रिप्शन का उपयोग करके सुरक्षित करता है।"
res_hi = qa_pipe(question="मैलशॉट फाइलों को कैसे सुरक्षित करता है?", context=context_hi)
print("\n[+] Hindi Test:")
print(f"    Answer : {res_hi['answer']}")
print(f"    Score  : {res_hi['score']:.4f}")

## 8. Export to ONNX Runtime with Quantization for MalShot Offline Edge Execution

In [ ]:
import subprocess
from pathlib import Path

onnx_export_path = "indic_malbert_threat_qa.onnx"
print(f"[*] Exporting fine-tuned model to ONNX: {onnx_export_path}...")

# Export using optimum / PyTorch ONNX exporter
!python -m transformers.onnx --model={output_dir} --feature=question-answering onnx_out/

import shutil
if Path("onnx_out/model.onnx").exists():
    shutil.copy("onnx_out/model.onnx", onnx_export_path)
    print(f"[+] Exported ONNX model successfully ({Path(onnx_export_path).stat().st_size / 1e6:.1f} MB)!")
else:
    print("[*] Model saved in PyTorch format. You can download the weights folder directly below.")

## 9. Launch Live Colab GPU QA Server (Cloudflare Tunnel)
Allows MalShot or any client to send extractive QA queries directly to this Colab GPU instance.

In [ ]:
import subprocess
import time
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import threading

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

qa_app = FastAPI(title="Indic-mALBERT Cybersecurity QA API")

class QARequest(BaseModel):
    question: str
    context: str

@qa_app.get("/health")
def qa_health():
    return {
        "status": "healthy",
        "model": "rohitsroch/indic-mALBERT-squad-v2-finetuned",
        "domain": "Cybersecurity & Threat Intelligence (Indic + English)",
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    }

@qa_app.post("/qa")
def answer_query(req: QARequest):
    t0 = time.perf_counter()
    res = qa_pipe(question=req.question, context=req.context)
    dt = (time.perf_counter() - t0) * 1000
    return {
        "answer": res["answer"],
        "confidence": round(float(res["score"]), 4),
        "start": res["start"],
        "end": res["end"],
        "latency_ms": round(dt, 2)
    }

def run_qa_server():
    uvicorn.run(qa_app, host="127.0.0.1", port=8001, log_level="warning")

qa_thread = threading.Thread(target=run_qa_server, daemon=True)
qa_thread.start()
time.sleep(2)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

import re
qa_tunnel_url = None
for line in tunnel_proc.stderr:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        qa_tunnel_url = m.group(0)
        break

print("=" * 75)
print("[+] LIVE INDIC-MALBERT QA COLAB TUNNEL ACTIVE!")
print(f"[+] Public Endpoint: {qa_tunnel_url}")
print("=" * 75)
print("\nQuery via Python Bridge:")
print(f"python colab_bridge.py --url {qa_tunnel_url} --qa --question 'What encryption is used?' --context 'MalShot uses AES-256-GCM.'")
